# T11: LLM Agents

`ToolAgent` (`conversational_toolkit/agents/tool_agent.py`) automates the loop from T10: call the LLM, run the requested tools, append their results, and repeat until the LLM answers without a tool call or `max_steps` LLM calls are used up. If a tool raises an exception, the error message goes back to the LLM instead of stopping the agent.

This notebook builds an agent with two tools, then the same agent with structured JSON output.

## Agent with two tools

The agent decides per query whether to use no tool, one tool, or several in sequence. The `Tool ... returned` lines show which tools ran.

`ToolAgent.answer` takes a `QueryWithContext` (query plus history) instead of a message list. `ask` wraps that call and prints the answer.

In [ ]:
import json
import sys

from loguru import logger

from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.agents.tool_agent import ToolAgent
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM

from ki_kompass_rag.baseline_rag import ask
from ki_kompass_rag.tool_agents import HalfBoldText, SumTwoNumbers

logger.remove()
logger.add(sys.stderr, level="WARNING")
# Print one line per tool call
logger.add(
    sys.stderr,
    level="DEBUG",
    filter="conversational_toolkit.agents.tool_agent",
    format="{message}",
)

llm = OpenAILLM(tools=[SumTwoNumbers(), HalfBoldText()], tool_choice="auto")
agent = ToolAgent(
    system_prompt="You are a helpful assistant. Answer briefly.", llm=llm, max_steps=5
)

queries = [
    "How are you?",
    "Sum 2 and 59.",
    "Bold every other word: 'This is an example of the half bold text tool in action.'",
    "Sum 10 and 20, then write one sentence about the result with every other word bolded.",
]
for query in queries:
    print(f"\n>>> {query}")
    await ask(agent, query)

## Structured output

`response_format` with a strict JSON Schema forces the final answer into an object with `answer` and `explanations`, so the result can be parsed reliably. Tool calls work as before; only the final message has to match the schema.

In [ ]:
answer_schema = {
    "type": "object",
    "properties": {
        "answer": {"type": "string", "description": "The answer in markdown."},
        "explanations": {
            "type": "array",
            "description": "Justifications for the answer.",
            "items": {"type": "string"},
            "minItems": 1,
            "maxItems": 3,
        },
    },
    "required": ["answer", "explanations"],
    "additionalProperties": False,
}

structured_llm = OpenAILLM(
    tools=[SumTwoNumbers(), HalfBoldText()],
    tool_choice="auto",
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "AnswerSchema",
            "schema": answer_schema,
            "strict": True,
        },
    },
)
structured_agent = ToolAgent(
    system_prompt="Answer like a pirate. Justify every tool use. Keep it short.",
    llm=structured_llm,
    max_steps=5,
)

for query in queries:
    print(f"\n>>> {query}")
    answer = await structured_agent.answer(QueryWithContext(query=query, history=[]))
    data = json.loads(answer.content[0].text or "")
    print(json.dumps(data, indent=2, ensure_ascii=False))

## Conversation

Earlier turns go into `history`. Only the user queries and the final answers are kept, not the intermediate tool calls.

In [ ]:
history = [
    LLMMessage(
        role=Roles.USER, content=[MessageContent(type="text", text=queries[-1])]
    ),
    answer,
]
follow_up = await structured_agent.answer(
    QueryWithContext(query="Make a joke about our conversation.", history=history)
)
data = json.loads(follow_up.content[0].text or "")
print(json.dumps(data, indent=2, ensure_ascii=False))